# nb07 — Judge competence × tie rate, and the applicability thresholds of the sorts

The simulation of nb05 describes the judge by β, which is not identifiable on
its own: the probability of an error depends on the product β·|Δs|, so doubling
the spread of true quality in the pool is equivalent to doubling β. The
scale-free quantity is the judge's **error rate** ε, the share of decisive
verdicts in which the judge prefers the weaker object. This is what can be
measured in practice, so the thresholds are reported in these units.

**Step 1 — the sweep.** The simulation of nb05 on a finer grid: 11 error
rates (2 to 18 errors per 100 decisive verdicts) × 9 tie rates (0 to 0.4) ×
3 input orders × 30 repetitions = 8,910 realizations, threshold tie model.
It takes about 2.5 hours (the published run took 8,330 s), so by default the
notebook reads the published sweep `results/20260918_120914_nb07_competence_sweep`.
Set `RUN_SWEEP = True` to recompute it. The sweep uses its own random seed,
so it is also an independent replication of the main run of nb05.

**Step 2 — the thresholds** (about 30 s). For every algorithm, error rate and
input order: the largest tie rate at which the mean Spearman correlation with
the true order still meets the criterion (ρ ≥ 0.95, 0.90, 0.85), interpolated
on the Fisher scale, with a bootstrap 95 % confidence interval over the
repetitions; and the critical error rate at which the criterion fails even
without ties.

Outputs go to `results/<time>_nb07_thresholds/`: `thresholds.csv`,
`critical_error_rates.csv`, the figures `fig_thresholds_<algorithm>`,
`report.txt`, `config.json`. The published run is
`results/20260918_210724_nb07_thresholds` (Figure 3, Figures A1–A2,
Tables A1–A3). The figures of the article carry the axis label of
ranklib 1.4.2 ("Judge error rate ε, %"), which this package uses.


In [ ]:
import os
from ranklib.experiments import run_simulation_experiment, run_threshold_analysis
from ranklib.simulation import beta_for_error, error_rate

ERROR_LEVELS = [2, 3, 4, 5, 6, 8, 10, 12, 14, 16, 18]      # errors per 100 decisive verdicts
TAUS = [round(0.05 * k, 2) for k in range(9)]              # 0 ... 0.4

PARAMS_SWEEP = dict(
    NB_ID="nb07_competence_sweep",
    QUICK=False,
    TAUS=TAUS,
    BETAS=[beta_for_error(e / 100.0) for e in ERROR_LEVELS],
    TIE_MODELS=["threshold"],
    ORDER_KINDS=["random", "pos", "neg"],
    REPS=30,
    N=400,
    ORDER_RHO=0.5,
    MASTER_SEED=20260918,          # differs from nb05: an independent replication
    RHO_THRESHOLDS=[0.95, 0.90, 0.85],
    ERROR_LEVELS=ERROR_LEVELS,     # recorded in config.json
)
RUN_SWEEP = False                  # True = recompute the sweep (about 2.5 hours)

for e, b in zip(ERROR_LEVELS, PARAMS_SWEEP["BETAS"]):
    print(f"{e:>3} errors per 100 -> beta = {b:7.3f}  (check: {error_rate(b) * 100:.2f})")


In [ ]:
if RUN_SWEEP:
    _long, _summary, _suit, sweep_run = run_simulation_experiment(PARAMS_SWEEP)
    SIM_RUN = sweep_run.dir
else:
    SIM_RUN = os.path.join("results", "20260918_120914_nb07_competence_sweep")
print("sweep:", SIM_RUN)


In [ ]:
PARAMS = dict(
    SIM_RUN=SIM_RUN,
    CRITERIA=[0.95, 0.90, 0.85],
    ALGORITHMS=["quicksort", "merge", "heapsort"],
    ORDER_KIND="random",
    N_BOOT=2000,               # bootstrap resamples for the confidence intervals
    BOOT_SEED=20260918,
    SHARED_AXES=True,          # common axes for the per-algorithm figures
)
thresholds, critical, run = run_threshold_analysis(PARAMS)
critical


In [ ]:
# Show the figures saved by this run
import glob, os
from IPython.display import Image, display
if run is not None:
    for p in sorted(glob.glob(os.path.join(run.dir, "*.png"))):
        print(os.path.basename(p))
        display(Image(p))
